# 01 — Structured Narrative Generation Demo

Implements the six-section `CaseNarrative` schema from Chapter 3 and generates a mock, fully-cited narrative from synthetic KYC/transaction/case-note data using an offline mock LLM function — no real API key required. Demonstrates that structured-output enforcement guarantees every section is present (even if empty), closing off the format-drift failure mode Course 1 Chapter 1 describes.

In [1]:
from dataclasses import dataclass, field
from typing import Optional
import json

@dataclass
class Citation:
    source_type: str  # "kyc_field" | "transaction_id" | "prior_case_id"
    source_id: str

@dataclass
class CitedClaim:
    text: str
    citations: list = field(default_factory=list)

@dataclass
class CaseNarrative:
    customer_kyc_overview: list
    alert_trigger_summary: list
    transaction_pattern_analysis: list
    historical_prior_case_context: list
    red_flags_identified: list
    recommendation: str
    recommendation_rationale: list
    confidence_by_section: dict

    def all_claims(self):
        return (
            self.customer_kyc_overview + self.alert_trigger_summary
            + self.transaction_pattern_analysis + self.historical_prior_case_context
            + self.red_flags_identified + self.recommendation_rationale
        )

print('Schema classes defined: Citation, CitedClaim, CaseNarrative.')

Schema classes defined: Citation, CitedClaim, CaseNarrative.


## Synthetic source data

A synthetic KYC profile, transaction history, alert detail, and retrieved prior case note — standing in for `get_kyc_profile`, `get_transaction_history`, `get_alert_trigger_detail` (Chapter 2's structured tools), and an Azure AI Search hit.

In [2]:
kyc_profile = {
    'customer_id': 'CUST-4471', 'occupation': 'Import/export consultant',
    'stated_income': 95000, 'risk_rating': 'MEDIUM', 'account_tenure_years': 3,
}

transaction_history = [
    {'transaction_id': 'TXN-88201', 'amount': 9800, 'date': '2026-06-01', 'counterparty': 'Overseas Trading Ltd'},
    {'transaction_id': 'TXN-88213', 'amount': 9700, 'date': '2026-06-02', 'counterparty': 'Overseas Trading Ltd'},
    {'transaction_id': 'TXN-88240', 'amount': 9650, 'date': '2026-06-03', 'counterparty': 'Overseas Trading Ltd'},
]

alert_detail = {
    'alert_id': 'ALERT-2026-0091', 'rule_fired': 'STRUCTURING_UNDER_10K',
    'trigger_transactions': ['TXN-88201', 'TXN-88213', 'TXN-88240'],
}

prior_case_notes = [
    {'case_id': 'CASE-2025-0033', 'text': 'Customer flagged for similar near-threshold wire pattern in Q4 2025; closed as false positive after invoice documentation confirmed legitimate trade financing.'},
]

print('Synthetic source data ready.')

Synthetic source data ready.


## Mock LLM generation function

A deterministic, offline stand-in for the Azure OpenAI structured-output call described in Chapter 3 — same schema contract, same citation discipline, no network call.

In [3]:
def mock_generate_narrative(kyc, txns, alert, case_notes) -> CaseNarrative:
    kyc_overview = [
        CitedClaim('Customer is an import/export consultant with 3 years of account tenure and a MEDIUM risk rating.',
                   [Citation('kyc_field', 'occupation'), Citation('kyc_field', 'risk_rating')]),
    ]
    alert_summary = [
        CitedClaim(f"Alert {alert['alert_id']} fired rule {alert['rule_fired']} on {len(alert['trigger_transactions'])} transactions.",
                   [Citation('transaction_id', t) for t in alert['trigger_transactions']]),
    ]
    pattern = [
        CitedClaim(f"Three transactions of ${t['amount']} were made to the same counterparty ({t['counterparty']}) on consecutive days, each just under the $10,000 reporting threshold.",
                   [Citation('transaction_id', t['transaction_id']) for t in txns])
        for t in txns[:1]
    ]
    history = [
        CitedClaim(f"A similar near-threshold pattern was previously reviewed and closed as a false positive ({cn['case_id']}) after trade-financing documentation was confirmed.",
                   [Citation('prior_case_id', cn['case_id'])])
        for cn in case_notes
    ]
    red_flags = [
        CitedClaim('Transaction amounts are consistently just under the $10,000 CTR threshold, a pattern consistent with structuring.',
                   [Citation('transaction_id', t['transaction_id']) for t in txns]),
    ]
    recommendation_rationale = [
        CitedClaim('Given the prior false-positive resolution for a similar pattern with documented trade financing, and no new red flags beyond amount clustering, escalation is not clearly warranted without further documentation review.',
                   [Citation('prior_case_id', case_notes[0]['case_id'])]),
    ]
    confidence = {
        'customer_kyc_overview': 0.95, 'alert_trigger_summary': 0.98,
        'transaction_pattern_analysis': 0.9, 'historical_prior_case_context': 0.6,
        'red_flags_identified': 0.85, 'recommendation_rationale': 0.55,
    }
    return CaseNarrative(
        customer_kyc_overview=kyc_overview, alert_trigger_summary=alert_summary,
        transaction_pattern_analysis=pattern, historical_prior_case_context=history,
        red_flags_identified=red_flags, recommendation='escalate',
        recommendation_rationale=recommendation_rationale, confidence_by_section=confidence,
    )

narrative = mock_generate_narrative(kyc_profile, transaction_history, alert_detail, prior_case_notes)
print('Narrative generated. Sections present:', list(narrative.__dataclass_fields__.keys()))

Narrative generated. Sections present: ['customer_kyc_overview', 'alert_trigger_summary', 'transaction_pattern_analysis', 'historical_prior_case_context', 'red_flags_identified', 'recommendation', 'recommendation_rationale', 'confidence_by_section']


In [4]:
for section in ['customer_kyc_overview', 'alert_trigger_summary', 'transaction_pattern_analysis',
                'historical_prior_case_context', 'red_flags_identified']:
    claims = getattr(narrative, section)
    conf = narrative.confidence_by_section[section]
    print(f"\n=== {section} (confidence={conf}) ===")
    for c in claims:
        cite_str = ', '.join(f"{cit.source_type}:{cit.source_id}" for cit in c.citations)
        print(f"  - {c.text}\n    [cites: {cite_str}]")

print(f"\nRecommendation: {narrative.recommendation}")
for c in narrative.recommendation_rationale:
    print(f"  rationale: {c.text}")

total_claims = len(narrative.all_claims())
uncited = [c for c in narrative.all_claims() if not c.citations]
print(f"\nTotal claims: {total_claims}, uncited claims: {len(uncited)}")
assert len(uncited) == 0, 'Every claim must carry at least one citation per Chapter 3 Rule 2.'
print('PASS: every claim in this narrative carries at least one citation.')


=== customer_kyc_overview (confidence=0.95) ===
  - Customer is an import/export consultant with 3 years of account tenure and a MEDIUM risk rating.
    [cites: kyc_field:occupation, kyc_field:risk_rating]

=== alert_trigger_summary (confidence=0.98) ===
  - Alert ALERT-2026-0091 fired rule STRUCTURING_UNDER_10K on 3 transactions.
    [cites: transaction_id:TXN-88201, transaction_id:TXN-88213, transaction_id:TXN-88240]

=== transaction_pattern_analysis (confidence=0.9) ===
  - Three transactions of $9800 were made to the same counterparty (Overseas Trading Ltd) on consecutive days, each just under the $10,000 reporting threshold.
    [cites: transaction_id:TXN-88201, transaction_id:TXN-88213, transaction_id:TXN-88240]

=== historical_prior_case_context (confidence=0.6) ===
  - A similar near-threshold pattern was previously reviewed and closed as a false positive (CASE-2025-0033) after trade-financing documentation was confirmed.
    [cites: prior_case_id:CASE-2025-0033]

=== red_flag